# Bài học 15: Pipeline Hoàn Chỉnh

Trong 2 bài trước, chúng ta đã xây dựng từng agent riêng lẻ. Giờ là lúc **nối tất cả lại** thành một pipeline hoàn chỉnh:

```
Chủ đề --> Nghiên cứu --> Dàn ý --> Viết bài --> Hình ảnh (tuỳ chọn) --> Bài viết hoàn thiện
```

Pipeline này chính là **mã nguồn sản phẩm thật**. Chúng ta sẽ import trực tiếp từ source code, không viết lại.

Điểm khác biệt so với việc chạy từng agent riêng lẻ:
- **Theo dõi qua database** — mỗi bước cập nhật trạng thái bài viết trong SQLite
- **Xử lý lỗi** — nếu bất kỳ bước nào thất bại, trạng thái chuyển sang `error` kèm thông báo chi tiết
- **Xuất file** — bài viết hoàn thiện tự động được lưu thành file `.md`

## Sơ đồ Pipeline

Đây là toàn bộ luồng xử lý với việc theo dõi trạng thái qua database:

```
create_article(topic)
      |
      v
  [queued]  -->  [researching/outlining]  -->  [writing]  -->  [enriching]  -->  [review]
                  Research Agent              Writer Agent     Image Agent        Xong!
                  + Outline Agent             (Grok-4)         (tuỳ chọn)
                  (Claude Sonnet)                              (Claude Sonnet)
                        |                         |                  |
                        v                         v                  v
                  Nếu lỗi: [error]         Nếu lỗi: [error]  Nếu lỗi: [error]
```

Tại mỗi bước:
1. Cập nhật trạng thái trong database
2. Chạy agent
3. Lưu kết quả và chuyển sang bước tiếp theo

Nếu bất kỳ bước nào thất bại, pipeline dừng lại và lưu `error_message` vào database.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../../output"))

from dotenv import load_dotenv
load_dotenv()

# Import pipeline thật từ sản phẩm của chúng ta!
# (Bạn đã thấy sys.path.insert lần đầu ở pipeline nhỏ trong Bài 12 — cùng cách làm ở đây)
from pipeline import run_content_pipeline
from db import create_article, get_article, init_db

init_db()

## `run_content_pipeline()` Làm Gì?

Hàm này là **trái tim** của sản phẩm. Nó chạy 4 bước:

1. **Nghiên cứu** — Gọi Research Agent, lưu ghi chú nghiên cứu
2. **Dàn ý** — Gọi Outline Agent với ghi chú nghiên cứu, lưu ContentOutline (JSON)
3. **Viết bài** — Gọi Writer Agent với dàn ý, lưu bài viết Markdown
4. **Làm giàu nội dung** (tuỳ chọn) — Nếu Image Agent tồn tại, tìm và chèn hình ảnh

Giữa mỗi bước, nó cập nhật database:
```python
update_article_status(article_id, "writing", outline=outline_json)
```

Khi gặp lỗi:
```python
update_article_status(article_id, "error", error_message=str(e))
```

Cuối cùng, bài viết được lưu ra file và trạng thái chuyển sang `review`.

## Xử Lý Lỗi: try / except

Chuyện gì xảy ra khi có sự cố? Một lệnh gọi API thất bại, tìm kiếm không trả về kết quả, hay mạng bị mất.

Python dùng **try/except** để xử lý lỗi một cách an toàn:

```python
try:
    # Thử chạy đoạn code này
    result = agent.run("...")
except Exception as e:
    # Nếu thất bại, làm cái này thay thế
    print(f"Đã xảy ra lỗi: {e}")
```

- Code bên trong `try` chạy bình thường
- Nếu bất kỳ dòng nào trong `try` thất bại, Python **nhảy** sang `except` ngay lập tức
- Biến `e` chứa thông báo lỗi
- Nếu không có try/except, toàn bộ chương trình sẽ bị crash

Pipeline của chúng ta bọc cả 4 bước agent trong try/except. Nếu bất kỳ bước nào thất bại:
1. Thông báo lỗi được lưu vào database (cột `error_message`)
2. Trạng thái bài viết chuyển sang `"error"`
3. Pipeline dừng lại và trả về `False`

Điều này có nghĩa là **một bài viết lỗi không làm crash cả batch** — các bài viết khác vẫn tiếp tục xử lý bình thường.

> **Chi phí:** Ô tiếp theo chạy toàn bộ pipeline (~$1-3 mỗi lần: Sonnet cho nghiên cứu + dàn ý, Grok cho viết bài). Mất khoảng 2-4 phút. Chỉ cần chạy một lần.

In [ ]:
topic = "10 Simple SEO Tips for Beginners"

# Tạo bản ghi bài viết trong database
article_id = create_article(topic)
print(f"Đã tạo bài viết #{article_id}\n")

# Chạy toàn bộ pipeline
success = run_content_pipeline(article_id, topic)
print(f"\nThành công: {success}")

In [ ]:
article = get_article(article_id)
print(f"Bài viết #{article['id']}")
print(f"  Chủ đề:     {article['topic']}")
print(f"  Trạng thái: {article['status']}")
print(f"  Số từ:      {article['word_count']}")
print(f"  File:       {article['output_file']}")

In [ ]:
if article.get("output_file"):
    with open(article["output_file"], "r", encoding="utf-8") as f:
        content = f.read()
    print(content[:2000] + "\n\n... (đã cắt bớt để hiển thị)")

## Bài Tập

Sử dụng dict `article` ở trên, viết code để kiểm tra chất lượng bài viết:

1. Số từ có trên 1000 không? In ra thông báo đạt/không đạt.
2. Bài viết đang ở trạng thái gì? (nên là `"review"` nếu thành công)
3. File output có tồn tại trên ổ đĩa không? (gợi ý: `os.path.exists(article['output_file'])`)

Nâng cao: Dùng `list_articles` từ `db` để xem tất cả bài viết trong database và đếm số bài ở mỗi trạng thái.

In [ ]:
# Bài tập: Viết code của bạn ở đây


## Chúc Mừng!

Bạn đã xây dựng xong một **pipeline tạo nội dung SEO bằng AI** hoàn chỉnh:

- **4 agent chuyên biệt**, mỗi agent sử dụng model và tools phù hợp nhất cho nhiệm vụ của nó
- **Theo dõi qua database** — giám sát trạng thái từ `queued` đến `review`
- **Xử lý lỗi** — pipeline dừng an toàn khi gặp lỗi
- **Xuất file** — bài viết tự động được lưu dưới dạng file Markdown

Code này chạy **y hệt** trong sản phẩm thật. Khi bạn chạy:
```bash
python output/cli.py create "topic"
```
Nó gọi đúng hàm `run_content_pipeline()` mà chúng ta vừa kiểm thử.

### Tiếp theo: Mô-đun 5

Trong mô-đun tiếp theo, chúng ta sẽ biến pipeline này thành một **sản phẩm hoàn chỉnh** với:
- Tầng database để lưu trữ bền vững
- Giao diện CLI (dòng lệnh)
- Giao diện chat (đội AI đàm thoại)
- Xử lý hàng loạt (tạo nhiều bài viết cùng lúc)